# Handling text data with Pandas

In [ ]:
# Load in pandas, and some other useful packages
# Note it is typical to load pandas as pd
import os, sys
import pandas as pd
import numpy as np
from calendar import month_name

from pathlib import Path

notebook_dir = os.getcwd()
data_dir = str(Path(notebook_dir).parents[1] / 'data_samples' / 'csv')

## Read/write data to/from Dataframes

Dataframes and Series are the core objects in Pandas. Think of a Dataframe as like an Excel spreadsheet, except we can do much more with a Dataframe and it is much faster than Excel! You can think of a Series as a single column in your spreadsheet.

We will start by loading a Dataframe from CSV. Let's take a look at the file first (just the first few lines)


In [ ]:
!head -n 5 ../../data_samples/csv/el_nino_3.4.csv

This text file has rows where data is separated by two spaces (in this case, each row gives the El Nino 3.4 Index for every month in a given year). 

Pandas is great at dealing with any data separated by a particular set of characters, including commas (of course!), and tabs.

NOTE: Here we are loading data that is separated by two blank spaces, so we have to specify the 'sep' parameter (as the default is comma separated). There are two alternatives to load this; use `sep='  '`, or `sep='\s+'`. The second one will deal with arbitrary numbers of blank spaces, so is more flexible. This is equivalent to using `delim_whitespace=True` in the input arguments

In [ ]:
column_names = ['YEAR'] + [month_name[n] for n in range(1, 13)]

df = pd.read_csv(os.path.join(data_dir, 'el_nino_3.4.csv'), 
                 header=None, 
                 sep='\s+', names=column_names)

In [ ]:
# .head() is a useful method to have a quick look at the data
df.head(n=5)

Note that is has dealt nicely with the NA values, and correctly interpreted them as numpy NaN objects.
In general, you can pass any other strings to read_csv that you want to be interepreted as NaN, via the `na_values` argument

Note that we can also deal with comments etc at the beginning of the file. Consider this file:

In [ ]:
!head -n 5 ../../data_samples/csv/el_nino_3.4_with_comments.csv

first lets see what Pandas does by default

In [ ]:
df_with_header_bad = pd.read_csv(os.path.join(data_dir, 'el_nino_3.4_with_comments.csv'), 
                 header=None, 
                 sep='\s+', names=column_names)

df_with_header_bad.head()

Solution: use the `skiprows` argument to skip the first two lines of the file

In [ ]:
# 
df_with_header = pd.read_csv(os.path.join(data_dir, 'el_nino_3.4_with_comments.csv'), 
                 header=None, 
                 sep='\s+', names=column_names, skiprows=2)

df_with_header.head()

read_csv has many other helpful options for loading data, look at the read_csv documentation for more! https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html

Once we have done our work, we can save the dataframe using .to_csv():

In [ ]:
df.to_csv('my_file.csv')

Pandas also supports other file formats other than .csv. The syntax is similar but with _csv replaced by the file format name

For example:

- parquet (.pq) -> pd.read_parquet(filename)
- json (.json) -> pd.read_json(filename)
- excel (.xlsx) -> pd.read_excel(filename)

## Describing the data

In [ ]:
# As noted above, .head(n) is very useful to have a quick look at the top rows
# .tail(n) shows the end rows
df.head(3)

In [ ]:
# len() works just like for Python lists and gives you the number of rows
print(len(df))

In [ ]:
# .describe() is a useful method to summarise the data
df.describe()

In [ ]:
# A single column of a Dataframe is called a Series
type(df['January'])

In [ ]:
# You can get a list of all column names from the .columns method
df.columns

In [ ]:
df.index

In [ ]:
# # sometimes, the columns in a dataframe are 'nested'; this means a column is accessed by specifying more than one name
nested_df = pd.read_csv(os.path.join(data_dir, 'nested_example.csv'), header=[0,1], index_col=0)

In [ ]:
nested_df.head()

In [ ]:
# # Access the columns by specyfing the right combination of labels
nested_df['precipitation']['min']

## Selecting and manipulating data

In [ ]:
df = df.rename({'YEAR': 'Year'}, axis=1)
df.head()

In [ ]:
# Select column
df['January']

In [ ]:
# Select based on row
df[df['Year'] == 1979]

In [ ]:
# Select based on an interval: in this case all rows for years greater or equal to 2020
df[df['Year'] >= 2020]

In [ ]:
# Select based on a list of items
df[df['Year'].isin([1990, 2021])]

In [ ]:
# Use ~ as a logical NOT operation. E.g. get all years that aren't 1990 or 2021
df[~df['Year'].isin([1990, 2021])].head()

In [ ]:
# Select based on multiple boolean conditions using | for 'OR' and & for 'AND'
df[ (df['Year'] >= 2020) | (df['Year'] < 1981)]

In [ ]:
# Select all years after 2000 with a negative index in December

df[ (df['Year'] > 2000) & (df['December'] < 0)]

## Modifying data

Create a new column; let's make it the sum of January and February

In [ ]:
df['Jan_plus_Feb'] = df['January'] + df['February']
df.head()

You can update based on your selection

e.g. imagine we are not confident in the data before 1985, let's set it to be NaN (='Not a Number'):

In [ ]:
updated_df = df.copy()

updated_df[updated_df['Year'] < 1985] = np.nan
updated_df.head(n=10)

# Notice that it has changed the years from integers to floats, since numpy's nan is a float

In [ ]:
# Now lets select only the non-null values using .notnull()

non_null_df = updated_df[updated_df['Year'].notnull()]
non_null_df.head()

In [ ]:
# We can also perform operations on the data in very intuitive ways, e.g. let's add 100 to every element in the January Series

df['January'] + 100

## Plotting data

In [ ]:
df

In [ ]:
# Pandas has some handy builtin plotting methods, for when you need to quickly examine your data

df.plot(title='My Plot', y=['January', 'July'], x='Year', ylabel='ENSO index', xlabel='Year')

In [ ]:
# Note that df.plot() returns a matplotlib axis, so you can tweak the plot in the normal matplotlib way

ax = df.plot(x='Year', y='January')
ax.set_ylabel('ENSO index')

In [ ]:
df['January'].plot(kind='hist', xlabel='ENSO index')

In [ ]:
# Smoothed version using Kernel Desnity Estimation (quite slow for large amounts of data)
# Note that for some reason xlabel doesn't work within the .plot command for this case
ax = df['January'].plot(kind='kde')
ax.set_xlabel('ENSO index')

## Indexes

In [ ]:
# We can also set Year as an Index. Note how the Year column data replaces the existing index.
df_indexed = df.set_index('Year').head()

In [ ]:
# This allows us to access data in a Series in a different way
df_indexed['January'][1979]

In [ ]:
# And allows us to access rows from a Dataframe in a more intuitive way, using .loc
df_indexed.loc[1979]

## Time Series Data

First load daily precipitation values. Note that the separator is a | 

In [ ]:
df_ts = pd.read_csv(os.path.join(data_dir, 'daily_precip.csv'), sep='|')

to unleash the full power of pandas, convert date columns to datetimes

In [ ]:
df_ts['date'] = pd.to_datetime(df_ts['date'])

This allows us to access intuitive date properties such as hour, day, month, day of year,

In [ ]:
df_ts['date'].dt.day_of_year

With time as an index, you can easily get things like year, month from the date

In [ ]:
df_ts = df_ts.set_index('date')
df_ts.index.month

Also, with a time index, we can more easily aggregate the time series data. E.g. lets get the weekly total

In [ ]:
df_ts.resample("7D").sum()

In [ ]:
# Or the yearly total
df_ts.resample("YS").sum()

## Grouping data

In [ ]:
# First load daily precipitation values. Note that the separator is a | 
df_ts = pd.read_csv(os.path.join(data_dir, 'daily_precip.csv'), sep='|')

In [ ]:
# Change date to a datetime column, and extact a new column with day of year
df_ts['date'] = pd.to_datetime(df_ts['date'])
df_ts['day_of_year'] = df_ts['date'].dt.day_of_year

In [ ]:
# You can use groupby with built in functions to calculate metrics 
# Lets get the average precipitation by day of year
# Note that we first remove the 'time' column, since we don't want to aggregate that

day_of_year_precip = df_ts[['day_of_year', 'precipitation']].groupby('day_of_year').mean()
day_of_year_precip.plot()

In [ ]:
# Or if you need to do several different summary statistics, agg() is more flexible
df_grouped = df_ts[['day_of_year', 'precipitation']].groupby('day_of_year').agg(['min', 'max', 'mean'])
df_grouped.head()

In [ ]:
# Note that aggregating in this way has created a nested DataFrame, so you can access the data like so
df_grouped['precipitation']['min'].plot()

## Creating Dataframes and Series

As well as loading from CSV, you can create your own dataframes from lists or dicts

In [ ]:
my_new_df = pd.DataFrame({'Column_1': [1,2,3,4,5], 'Column_2': ['A', 'B', 'A', 'C', 'A'], 'Column_3': [0.1, 0.3, -1,0.1, 0.2]})
my_new_df.head()

An equivalent way to do this is can also use a list of dicts

In [ ]:
my_new_df = pd.DataFrame([{'Column_1': 1, 'Column_2': 'A',  'Column_3': 0.1},
                          {'Column_1': 2, 'Column_2': 'B',  'Column_3': 0.3},
                          {'Column_1': 3, 'Column_2': 'A',  'Column_3': -1},
                          {'Column_1': 4, 'Column_2': 'C',  'Column_3': 0.1},
                          {'Column_1': 5, 'Column_2': 'A',  'Column_3': 0.2}])

my_new_df.head()

In [ ]:
# Now lets create a Series
my_new_series = pd.Series({'series_name': [1,2,3,4,5]})

## Exercises

Using observations from the Radcliffe Observatory

1) Load the dataframe in `example_data/Oxford-Radcliffe-daily-data-to-Dec-2023.csv`. Inspect the file to check what information it contains

2) Subselect only these columns: `Daily Tmean °C`, `Rainfall mm raw incl traces`,`YYYY`, `MM`, `DD`

3) Combine `YYYY`, `MM`, `DD` into a single date column called `Dates` with the format 'YYYY-MM-DD'. Convert this to a datetime column

4) How many dates have Null values for `Rainfall mm raw incl traces`?

5) Convert `Daily Tmean °C` to Kelvin; rename the column appropriately.

6) Calculate the average temperature (in Kelvin) for each year

7) Calculate the mean and standard deviation of the temperature by month

8) Plot the distribution of the temperature values

OPTIONAL EXTRAS:

9) Plot the distribution of rainfall values. What kind of shape does it have for extremely high rainfall? What kind of distribution might it be? If you are interested, try fitting a distribution to see what fits the best.

## Additional functionality

For those who have time / interest, here are some other methods that may be useful, although you can do most things with what we have already covered above

In [ ]:
# Select rows and columns simultaneously
df.loc[df['Year'] > 2000, ['November', 'December']]

In [ ]:
# You can even use the ':' syntax to specify a range of columns
df.loc[df['Year'] > 2000, 'October':'December']

In [ ]:
# This also allows you to e.g. find the year with the smallest value
df.loc[df['January'].argmin()]

In [ ]:
# select based on the index of the row/ column instead

df.iloc[0:5,0:3]

In [ ]:
# Use stack to get all monthly data in one column
df.loc[:,'January':'December'].stack().head()


In [ ]:
# Change month from a column to a variable
df.melt(id_vars=['Year'], var_name='Month')

## Joining data

There is an excellent in-depth tutorial on this on the Real Python site: https://realpython.com/pandas-merge-join-and-concat/

We will give a brief overview here

In [ ]:
# First lets create some dummy data, for temperature and precipitation recorded at some fictitious weather stations.
# Note that the location IDs are deliberately not the same in the two dataframes
df_temp = pd.DataFrame({'Location_ID': range(100), 'Temperature': 283*np.random.normal(1,0.1,100)})
df_precip = pd.DataFrame({'Location_ID': range(40,140), 'Precip': np.random.lognormal(mean=1.0, sigma=1.0, size=100)})

print(df_temp.head())
print(df_precip.head())

In [ ]:
# First lets try an inner join on location ID: this will only return results where the Location_ID column matches in both dataframes
inner_merged = pd.merge(df_temp, df_precip, how='inner', on='Location_ID')

print(inner_merged.head())

print('\nNumber of rows in joined dataframe:', inner_merged.shape)

In [ ]:
# First lets try an inner join on location ID: this will only return results where the Location_ID column matches in both dataframes
inner_merged = pd.merge(df_temp, df_precip, how='inner', on='Location_ID')

print(inner_merged.head())

print('\nNumber of rows in joined dataframe:', inner_merged.shape)

In [ ]:
# Now lets try a full outer join on location ID: this will return results for all values of Location_ID in both dataframes
# Where there is a mismatch in Location_ID values, the value will be NaN in the Dataframe
inner_merged = pd.merge(df_temp, df_precip, how='outer', on='Location_ID')

print(inner_merged.head())

print('\nNumber of rows in joined dataframe:', inner_merged.shape)

In [ ]:
# Now lets try a left outer join on location ID: this will return results for all values of Location_ID in df_temp (so 100 in total)
inner_merged = pd.merge(df_temp, df_precip, how='left', on='Location_ID')

print(inner_merged.head())

print('\nNumber of rows in joined dataframe:', inner_merged.shape)

In [ ]:
# Similarly, a 'right' join returns results for all values of Location_ID in df_precip (100 in total)
inner_merged = pd.merge(df_temp, df_precip, how='right', on='Location_ID')

print(inner_merged.tail(10))

print('\nNumber of rows in joined dataframe:', inner_merged.shape)